# 02_01 · Preprocesado — AI4I Predictive Maintenance


In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

AI4I_PATH = '../data/raw/ai4i2020.csv'


## 1. AI4I — Encoding y Feature Engineering

### Encoding categórico
`Type` (L/M/H) se convierte a entero (0/1/2) con `LabelEncoder`.
Este encoding ordinal es apropiado aquí porque L→M→H representa calidad creciente
y el umbral de fallo OSF escala con el tipo: L=11.000, M=12.000, H=13.000 Nm·min.

### Features de ingeniería — justificación física

Cada feature nueva reproduce exactamente la condición de un modo de fallo:

| Feature | Cálculo | Modo de fallo que captura |
|---|---|---|
| `Power [W]` | Torque × ω (vel. angular) | **PWF**: fallo si Power < 3500 W o > 9000 W |
| `Temp_diff [K]` | Process T − Air T | **HDF**: fallo si Temp_diff < 8.6 K y velocidad < 1380 rpm |
| `Wear_torque` | Tool wear × Torque | **OSF**: fallo si Wear_torque > umbral según tipo |
| `wear_ratio` | Tool wear / umbral_TWF[Type] | **TWF**: ratio de desgaste normalizado por tipo |

Sin estas features, el modelo tendría que descubrir estas interacciones por sí solo —
dándoselas ya calculadas aceleramos el aprendizaje y mejoramos la interpretabilidad.


In [2]:
ai4i = pd.read_csv(AI4I_PATH)

ai4i['Type_enc'] = LabelEncoder().fit_transform(ai4i['Type'])

# Umbrales de desgaste por tipo (AI4I dataset description)
TWF_MIN = {'L': 200, 'M': 220, 'H': 240}

# Feature Engineering
ai4i['Power [W]']     = ai4i['Torque [Nm]'] * ai4i['Rotational speed [rpm]'] * 2 * np.pi / 60
ai4i['Temp_diff [K]'] = ai4i['Process temperature [K]'] - ai4i['Air temperature [K]']
ai4i['Wear_torque']   = ai4i['Tool wear [min]'] * ai4i['Torque [Nm]']
ai4i['wear_ratio']    = ai4i.apply(lambda r: r['Tool wear [min]'] / TWF_MIN[r['Type']], axis=1)

feature_cols = ['Type_enc', 'Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
                'Power [W]', 'Temp_diff [K]', 'Wear_torque', 'wear_ratio']

X_ai4i = ai4i[feature_cols]
y_ai4i = ai4i['Machine failure']

print('Features:', feature_cols)
print(f'Clase 0: {(y_ai4i==0).sum()} | Clase 1: {(y_ai4i==1).sum()}')


Features: ['Type_enc', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Power [W]', 'Temp_diff [K]', 'Wear_torque', 'wear_ratio']
Clase 0: 9661 | Clase 1: 339


## 2. AI4I — Escalado y Split estratificado

### ¿Por qué StandardScaler?
Los árboles de decisión (RF, LGBM, XGB) son **invariantes al escalado** — no lo necesitan.
Sin embargo, standardizamos igualmente por dos razones:
- El **meta-modelo del stacking** es una Regresión Logística, que sí es sensible a la escala
- El **SVM** con kernel RBF requiere escalado obligatoriamente para funcionar correctamente
- Usar el mismo escalado para todos los modelos simplifica el pipeline de producción

### ¿Por qué stratify=y_ai4i?
Con solo 3.4% de fallos, un split aleatorio podría terminar con proporciones muy distintas
en train y test (e.g., 2% vs 5%). La **estratificación garantiza** que la tasa de fallo
sea idéntica en ambos conjuntos → métricas comparables y reproducibles.

### ¿Por qué 80/20?
Con 10.000 muestras y 339 fallos:
- 80% train → 271 fallos para aprender (mínimo razonable para árboles y ensembles)
- 20% test → 68 fallos para evaluar (suficientes para Recall estable con ±5% error)
- Un split 90/10 dejaría solo 34 fallos en test → intervalos de confianza muy amplios


In [3]:
scaler_ai4i = StandardScaler()
X_scaled = scaler_ai4i.fit_transform(X_ai4i)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_ai4i, test_size=0.2, random_state=42, stratify=y_ai4i)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Failure rate train: {y_train.mean():.2%} | test: {y_test.mean():.2%}')


Train: (8000, 10) | Test: (2000, 10)
Failure rate train: 3.39% | test: 3.40%


## 3. Guardar datos procesados AI4I

Guardamos `ai4i_featured.csv` y actualizamos `splits.pkl` con la clave `ai4i`.


In [4]:
import pickle, os
os.makedirs('../data/processed', exist_ok=True)

pkl_path = '../data/processed/splits.pkl'
splits = {}
if os.path.exists(pkl_path):
    with open(pkl_path, 'rb') as f:
        splits = pickle.load(f)

splits['ai4i'] = (X_train, X_test, y_train, y_test)
with open(pkl_path, 'wb') as f:
    pickle.dump(splits, f)

ai4i.to_csv('../data/processed/ai4i_featured.csv', index=False)
print('AI4I guardado OK.')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')


AI4I guardado OK.
Train: (8000, 10) | Test: (2000, 10)
